In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import physical_constants
from scipy.interpolate import interp1d

In [2]:
cube4="/Users/cpi/TRASH/baderT2T4/T4/charge.cube"
cube2="/Users/cpi/TRASH/baderT2T4/T2/charge.cube"
slab4="/Users/cpi/TRASH/baderT2T4/T4/slab.cube"
slab2="/Users/cpi/TRASH/baderT2T4/T2/slab.cube"
mol4="/Users/cpi/TRASH/baderT2T4/T4/mol.cube"
mol2="/Users/cpi/TRASH/baderT2T4/T2/mol.cube"

In [3]:
def read_cube_full(filename):
    """
    Read a Gaussian cube file.

    Returns
    -------
    header_lines : list[str]
        Original header lines up to atom list (used for writing)
    atom_lines : list[str]
        Atom specification lines
    rho : ndarray (nx, ny, nz)
        Volumetric data (e / bohr^3)
    grid_shape : tuple
        (nx, ny, nz)
    """
    with open(filename) as f:
        lines = f.readlines()

    # First two comment lines
    header_lines = lines[:2]

    # Origin + grid
    natoms = int(lines[2].split()[0])
    grid_lines = lines[2:6]

    header_lines += grid_lines

    nx = int(grid_lines[1].split()[0])
    ny = int(grid_lines[2].split()[0])
    nz = int(grid_lines[3].split()[0])

    atom_lines = lines[6:6 + natoms]

    data_lines = lines[6 + natoms:]

    raw = []
    for line in data_lines:
        raw.extend(map(float, line.split()))

    rho = np.array(raw).reshape((nx, ny, nz))

    return header_lines, atom_lines, rho, (nx, ny, nz)


In [4]:
def write_cube(filename, header_lines, atom_lines, rho):
    """
    Write a Gaussian cube file.
    """
    nx, ny, nz = rho.shape

    with open(filename, "w") as f:
        # Header
        for line in header_lines:
            f.write(line)

        # Atom list
        for line in atom_lines:
            f.write(line)

        # Volumetric data: 6 values per line (Gaussian convention)
        flat = rho.flatten()
        for i in range(0, len(flat), 6):
            f.write(" ".join(f"{val: .6e}" for val in flat[i:i+6]) + "\n")


In [ ]:
def read_cube_charge(filename):
    """
    Read Gaussian cube charge density.

    Returns
    -------
    rho : ndarray (nx, ny, nz)
        Charge density in e / bohr^3
    z_bohr : ndarray (nz,)
        z coordinates in bohr
    dx, dy, dz : float
        Grid spacings in bohr
    """
    with open(filename) as f:
        lines = f.readlines()

    header = lines[2:6]

    natoms = int(header[0].split()[0])

    nx, ax, _, _ = map(float, header[1].split())
    ny, _, ay, _ = map(float, header[2].split())
    nz, _, _, az = map(float, header[3].split())

    nx, ny, nz = int(nx), int(ny), int(nz)

    data_start = 6 + natoms

    raw = []
    for line in lines[data_start:]:
        raw.extend(map(float, line.split()))

    rho = np.array(raw).reshape((nx, ny, nz))

    dx, dy, dz = ax, ay, az          # bohr
    z_bohr = np.arange(nz) * dz      # bohr

    return rho, z_bohr, dx, dy, dz


In [ ]:
def cumulative_charge_z(rho, dx, dy, dz, z, zmin=None, zmax=None):
    """
    Cumulative in-plane integrated charge along z.

    All quantities in atomic units (bohr).
    Output charge in electrons.
    """
    mask = np.ones_like(z, dtype=bool)
    if zmin is not None:
        mask &= z >= zmin
    if zmax is not None:
        mask &= z <= zmax

    z_sel = z[mask]
    rho_sel = rho[:, :, mask]

    # In-plane integral: e / bohr
    q_xy = rho_sel.sum(axis=(0, 1)) * dx * dy

    # Cumulative z integral: electrons
    Qz = np.cumsum(q_xy) * dz

    return z_sel, Qz


In [ ]:
rho4, z_bohr, dx, dy, dz = read_cube_charge(cube4)
rho2, z2_bohr, dx2, dy2, dz2 = read_cube_charge(cube2)
slab4, zs_bohr, dxs, dys, dzs = read_cube_charge(slab4)
slab2, zs2_bohr, dxs2, dys2, dzs2 = read_cube_charge(slab2)

assert np.allclose(z_bohr, z2_bohr)
assert dx == dx2 and dy == dy2 and dz == dz2


In [ ]:
z4, Q4 = cumulative_charge_z(rho4, dx, dy, dz, z_bohr)
z2, Q2 = cumulative_charge_z(rho2, dx, dy, dz, z_bohr)
zs4, Qs4 = cumulative_charge_z(slab4, dx, dy, dz, z_bohr)
zs2, Qs2 = cumulative_charge_z(slab2, dx, dy, dz, z_bohr)

print("Total charge T4Au:", Q4[-1])
print("Total charge T2Au:", Q2[-1])
print("Total charge slab 4:", Qs4[-1])
print("Total charge slab 2:", Qs2[-1])


In [ ]:
from scipy.constants import physical_constants
bohr_to_ang = physical_constants["Bohr radius"][0] * 1e10  # Å
# Optional plotting limits (bohr)
zmin = 19.5/bohr_to_ang    # e.g. 10.0
zmax = 20.75/bohr_to_ang    # e.g. 35.0



# Determine plotting bounds
zmin_plot = z1.min() if zmin is None else zmin
zmax_plot = z1.max() if zmax is None else zmax

mask4 = (z4 >= zmin_plot) & (z4 <= zmax_plot)
mask2 = (z2 >= zmin_plot) & (z2 <= zmax_plot)
masks4 = (zs4 >= zmin_plot) & (zs4 <= zmax_plot)
masks2 = (zs2 >= zmin_plot) & (zs2 <= zmax_plot)

plt.figure(figsize=(7, 5))
ax = plt.gca()

ax.plot(z4[mask4] * bohr_to_ang, Q4[mask4], lw=2, label="T4@Au(111)")
ax.plot(z2[mask2] * bohr_to_ang, Q2[mask2], lw=2, label="T2@Au(111)")
ax.plot(zs4[masks4] * bohr_to_ang, Qs4[masks4], lw=2, label="Au(111)")
ax.axhline(12150, color="black", linestyle="--", lw=1.5, label="Q of neutral Au(111)")
ax.set_xlabel("z (Å)")
ax.set_ylabel("Cumulative charge (e)")

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Find Z given Q

In [ ]:
def z_at_charge(z, Q, Q_target):
    """
    Find the smallest z such that Q(z) >= Q_target.

    Parameters
    ----------
    z : ndarray
        z coordinates (bohr)
    Q : ndarray
        cumulative charge (electrons), non-decreasing
    Q_target : float
        target charge (electrons)

    Returns
    -------
    z_target : float
        z value (bohr)
    """

    # Safety checks
    if Q_target <= Q[0]:
        return z[0]

    if Q_target >= Q[-1]:
        return z[-1]

    # Enforce strict monotonicity for interpolation
    dQ = np.diff(Q)
    mask = np.hstack(([True], dQ > 0))

    z_mono = z[mask]
    Q_mono = Q[mask]

    # Interpolate inverse function z(Q)
    z_of_Q = interp1d(
        Q_mono,
        z_mono,
        kind="linear",
        bounds_error=False,
        fill_value=(z_mono[0], z_mono[-1]),
        assume_sorted=True,
    )

    z_est = float(z_of_Q(Q_target))

    # Enforce "first z such that Q >= Q_target"
    idx = np.searchsorted(Q, Q_target, side="left")
    z_left = z[idx]

    return min(z_est, z_left)


In [ ]:
Q_target4 = 12152  # electrons
Q_target2 = 12151

z_Q4 = z_at_charge(z4, Q4, Q_target4)
z_Q2 = z_at_charge(z2, Q2, Q_target2)

print("T4Au: Q =", Q_target4, "at z =", z_Q4*bohr_to_ang, "A")
print("T2Au: Q =", Q_target2, "at z =", z_Q2*bohr_to_ang, "A")


# Find Q given Z

In [ ]:
def charge_at_z(z, Q, z_target):
    """
    Compute cumulative charge Q(z_target) using interpolation.

    Parameters
    ----------
    z : ndarray
        z coordinates (bohr)
    Q : ndarray
        cumulative charge (electrons), non-decreasing
    z_target : float
        target z (bohr)

    Returns
    -------
    Q_target : float
        cumulative charge at z_target (electrons)
    """

    # Bounds handling
    if z_target <= z[0]:
        return Q[0]

    if z_target >= z[-1]:
        return Q[-1]

    # Interpolator Q(z)
    Q_of_z = interp1d(
        z,
        Q,
        kind="linear",
        bounds_error=False,
        fill_value=(Q[0], Q[-1]),
        assume_sorted=True,
    )

    return float(Q_of_z(z_target))


In [ ]:
z_found4 = z_Q4 # bohr
Q_check4 = charge_at_z(z4, Q4, z_found4)

print("Requested Q:", Q_target4)
print("Found z:", z_found4/bohr_to_ang)
print("Q at found z:", Q_check4)

# Charge density difference

In [5]:
cube_files = [
    cube4,
    slab4,
    mol4
]

# Read first cube (reference)
header, atoms, rho_ref, shape = read_cube_full(cube_files[0])

rho_diff = rho_ref.copy()

# Subtract all others
for fname in cube_files[1:]:
    _, _, rho_i, shape_i = read_cube_full(fname)
    assert shape_i == shape
    rho_diff -= rho_i


In [6]:
write_cube("diff4.cube", header, atoms, rho_diff)